[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/24_ensemble_learning/02_stacking_and_blending.ipynb)

# Bài 02 — Stacking và Blending

**🎯 Mục tiêu**
- Ý tưởng stacking: một **meta-model** học cách kết hợp dự đoán của các base model.
- Vì sao phải dùng dự đoán **out-of-fold** — stacking ngây thơ rò rỉ và chọn nhầm model overfit.
- Stacking from scratch và với `StackingRegressor` / `StackingClassifier`; `passthrough`.
- **Blending** (holdout) và so sánh; lời khuyên thực hành.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/24_ensemble_learning

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (HistGradientBoostingClassifier, HistGradientBoostingRegressor, RandomForestClassifier,
                              RandomForestRegressor, StackingClassifier, StackingRegressor)
from sklearn.linear_model import LinearRegression, LogisticRegression, RidgeCV
from sklearn.metrics import mean_squared_error, roc_auc_score
from sklearn.model_selection import KFold, cross_val_predict, train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

from ml_course.visualization import set_style

set_style()


def rmse(a, b):
    return np.sqrt(mean_squared_error(a, b))

## 1. Ý tưởng

Voting dùng trọng số cố định. **Stacking** (Wolpert, 1992) để một model cấp 2 (**meta-learner**) học cách kết hợp:

```
            ┌─ Ridge ──────┐
X ──────────┼─ KNN ────────┼──► [ŷ₁, ŷ₂, …, ŷ_M] ──► meta-model ──► ŷ
            ├─ SVR ────────┤        (meta-feature)
            └─ HistGB ─────┘
```

Câu hỏi then chốt: meta-model học trên dự đoán **nào** của base model?

In [ ]:
house = pd.read_csv("../../data/sample/house_prices.csv")
y = house.pop("price_billion_vnd").to_numpy()
num_cols = ["area_m2", "bedrooms", "bathrooms", "age_years", "distance_to_center_km"]


def prep():
    return ColumnTransformer([("num", StandardScaler(), num_cols), ("cat", OneHotEncoder(handle_unknown="ignore"), ["district"]),
                              ("bin", "passthrough", ["has_garden"])], sparse_threshold=0)


X_tr, X_te, y_tr, y_te = train_test_split(house, y, test_size=0.3, random_state=0)
base = {"Ridge": make_pipeline(prep(), RidgeCV()), "KNN": make_pipeline(prep(), KNeighborsRegressor(10)),
        "Random Forest": make_pipeline(prep(), RandomForestRegressor(300, random_state=0, n_jobs=-1)),
        "HistGB": make_pipeline(prep(), HistGradientBoostingRegressor(random_state=0)),
        "Decision Tree": make_pipeline(prep(), DecisionTreeRegressor(random_state=0)),
        "SVR": make_pipeline(prep(), SVR(C=10))}
names = list(base)
kf = KFold(5, shuffle=True, random_state=0)
P_in, P_oof, P_te = {}, {}, {}
for name, model in base.items():
    model.fit(X_tr, y_tr)
    P_in[name] = model.predict(X_tr)                                    # dự đoán trên chính dữ liệu đã train
    P_te[name] = model.predict(X_te)
    P_oof[name] = cross_val_predict(model, X_tr, y_tr, cv=kf)          # mỗi điểm được dự đoán bởi model KHÔNG thấy nó
summary = pd.DataFrame({"RMSE in-sample": {n: rmse(y_tr, P_in[n]) for n in names},
                        "RMSE out-of-fold": {n: rmse(y_tr, P_oof[n]) for n in names},
                        "RMSE test": {n: rmse(y_te, P_te[n]) for n in names}}).round(4)
summary

In [ ]:
ax = summary.plot(kind="bar", figsize=(11, 3.8), rot=0, color=["C1", "C0", "C3"])
ax.set_ylabel("RMSE (tỉ đồng)")
ax.set_title("In-sample quá lạc quan (cây quyết định = 0); out-of-fold sát với test")
plt.show()

Cây quyết định không giới hạn có RMSE **in-sample = 0** (thuộc lòng dữ liệu) nhưng test tệ nhất. Random Forest, HistGB cũng có in-sample thấp hơn nhiều so với out-of-fold.

## 2. Stacking ngây thơ: rò rỉ

Nếu meta-model học trên dự đoán **in-sample**, nó thấy cây quyết định "hoàn hảo" và dồn hết trọng số cho nó:

In [ ]:
A_in = np.column_stack([P_in[n] for n in names])
A_oof = np.column_stack([P_oof[n] for n in names])
A_te = np.column_stack([P_te[n] for n in names])
results = {}
for label, A in [("ngây thơ (in-sample)", A_in), ("out-of-fold", A_oof)]:
    meta = LinearRegression(positive=True).fit(A, y_tr)                 # trọng số không âm: dễ diễn giải, ổn định
    results[label] = rmse(y_te, meta.predict(A_te))
    print(f"meta-model học trên dự đoán {label:22s} trọng số = {dict(zip(names, meta.coef_.round(3)))}")
    print(f"{'':49s} RMSE test = {results[label]:.4f}")
print(f"\ntham chiếu: base model tốt nhất (test) = {summary['RMSE test'].min():.4f} | trung bình đều 6 model = {rmse(y_te, A_te.mean(1)):.4f}")

- **Ngây thơ**: toàn bộ trọng số cho cây quyết định → test RMSE tệ nhất. Meta-model bị lừa bởi dự đoán đã "nhìn thấy" đáp án.
- **Out-of-fold**: dự đoán của mỗi điểm đến từ model không train trên nó → giống dự đoán trên dữ liệu mới → meta-model học trọng số thật sự có ích (SVR, HistGB, Ridge lớn; cây đơn gần 0) → **tốt hơn** model tốt nhất và trung bình đều.

**Quy trình stacking chuẩn:**
1. Với mỗi base model: `cross_val_predict` trên train → meta-feature OOF.
2. Train meta-model trên meta-feature OOF.
3. Fit lại mỗi base model trên **toàn bộ** train; tại test: base model dự đoán → meta-model kết hợp.

## 3. Với scikit-learn

`StackingRegressor`/`StackingClassifier` làm đúng quy trình trên (tham số `cv`). Với phân loại, meta-feature là `predict_proba` (mặc định `stack_method="auto"`).

In [ ]:
st = StackingRegressor([(n, base[n]) for n in names], final_estimator=RidgeCV(), cv=kf, n_jobs=-1)
t0 = time.perf_counter()
st.fit(X_tr, y_tr)
print(f"StackingRegressor: RMSE test = {rmse(y_te, st.predict(X_te)):.4f} | thời gian fit = {time.perf_counter() - t0:.1f}s "
      f"(6 base model × (5 fold + 1 lần fit cuối))")
print("hệ số meta-model (RidgeCV):", dict(zip(names, st.final_estimator_.coef_.round(3))))

Kết quả khớp bản tự cài đặt ở mục 2. Meta-model nên **đơn giản** (tuyến tính, có regularization hoặc trọng số không âm) vì meta-feature ít và tương quan rất mạnh với nhau (mọi base model đều dự đoán cùng một giá). `passthrough=True` đưa thêm feature gốc vào meta-model — feature gốc phải là số (ở đây cột `district` dạng chuỗi sẽ gây lỗi, cần tiền xử lý riêng); hiếm khi đáng công.

Phân loại — dữ liệu khách hàng rời bỏ:

In [ ]:
churn = pd.read_csv("../../data/sample/customer_churn.csv")
yc = churn.pop("churn").to_numpy()
Xc = churn.drop(columns="customer_id")
cat_c = ["contract", "internet_service", "payment_method"]
num_c = ["tenure_months", "monthly_charges", "total_charges", "support_calls", "n_products"]


def prep_c():
    return ColumnTransformer([("num", StandardScaler(), num_c), ("cat", OneHotEncoder(handle_unknown="ignore"), cat_c),
                              ("bin", "passthrough", ["senior"])], sparse_threshold=0)


Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3, random_state=0, stratify=yc)
base_c = [("logreg", make_pipeline(prep_c(), LogisticRegression(max_iter=2000))),
          ("knn", make_pipeline(prep_c(), KNeighborsClassifier(25))),
          ("rf", make_pipeline(prep_c(), RandomForestClassifier(300, min_samples_leaf=5, random_state=0, n_jobs=-1))),
          ("hgb", make_pipeline(prep_c(), HistGradientBoostingClassifier(random_state=0)))]
for name, m in base_c:
    print(f"{name:8s} ROC-AUC test = {roc_auc_score(yc_te, m.fit(Xc_tr, yc_tr).predict_proba(Xc_te)[:, 1]):.4f}")
stack_c = StackingClassifier(base_c, final_estimator=LogisticRegression(), cv=5, n_jobs=-1).fit(Xc_tr, yc_tr)
print(f"{'stacking':8s} ROC-AUC test = {roc_auc_score(yc_te, stack_c.predict_proba(Xc_te)[:, 1]):.4f}")
print("trọng số meta-model (LogReg trên xác suất):", dict(zip([n for n, _ in base_c], stack_c.final_estimator_.coef_[0].round(2))))

Trên dữ liệu churn, Logistic Regression đã gần như tối ưu (quan hệ gần tuyến tính trong không gian logit) — stacking chỉ ngang hoặc nhỉnh hơn **rất ít**. Stacking không phải phép màu: lợi ích lớn khi base model **mạnh tương đương** và **khác nhau**; nhỏ khi một model đã chiếm ưu thế.

## 4. Blending

**Blending**: tách một phần train làm **holdout**; base model train trên phần còn lại, meta-model train trên dự đoán holdout. Đơn giản, không cần CV, nhưng lãng phí dữ liệu (base model và meta-model đều thấy ít dữ liệu hơn).

In [ ]:
X_a, X_b, y_a, y_b = train_test_split(X_tr, y_tr, test_size=0.3, random_state=1)
B_hold = np.column_stack([base[n].fit(X_a, y_a).predict(X_b) for n in names])
B_test = np.column_stack([base[n].predict(X_te) for n in names])
blend = LinearRegression(positive=True).fit(B_hold, y_b)
print(f"blending (holdout 30%): RMSE test = {rmse(y_te, blend.predict(B_test)):.4f} | stacking OOF = {results['out-of-fold']:.4f}")
print("trọng số blending:", dict(zip(names, blend.coef_.round(3))))

| | Stacking (OOF) | Blending (holdout) |
|---|---|---|
| Dữ liệu cho meta-model | toàn bộ train (OOF) | chỉ holdout |
| Base model cuối cùng | train trên toàn bộ | train trên phần còn lại (hoặc fit lại) |
| Chi phí | K lần fit mỗi base model | 1 lần |
| Rủi ro | ít | trọng số nhiễu nếu holdout nhỏ |

## 5. Lời khuyên thực hành

- **Đánh giá cả pipeline** trên test/CV ngoài cùng — tuning base model và meta-model trên cùng OOF dễ lạc quan (nested CV, chương 15).
- Meta-model đơn giản; base model **đa dạng** (tuyến tính, láng giềng, kernel, cây).
- Chuỗi thời gian: OOF phải theo thời gian (`TimeSeriesSplit`) — KFold xáo trộn rò rỉ tương lai.
- Chi phí: dự đoán cần chạy **mọi** base model → độ trễ, bộ nhớ, bảo trì (bài 03). Stacking hay thắng Kaggle vì vài phần nghìn quan trọng; production thì cân nhắc.

## 🧠 Kiểm tra nhanh

1. Vì sao meta-model phải học trên dự đoán out-of-fold?
2. Trong stacking ngây thơ, model nào được ưu ái? Vì sao?
3. Vì sao meta-model nên đơn giản?
4. Blending khác stacking ở đâu? Ưu/nhược?
5. Với chuỗi thời gian, tạo meta-feature thế nào?

## 📌 Tóm tắt

- Stacking: meta-model kết hợp dự đoán; **bắt buộc OOF** để tránh rò rỉ.
- `StackingRegressor/Classifier` tự làm OOF; meta-model tuyến tính, trọng số không âm/regularized.
- Blending: holdout, đơn giản hơn nhưng tốn dữ liệu; lợi ích thật phụ thuộc độ đa dạng và chất lượng base model.

## 📚 Tài liệu tham khảo

- Wolpert (1992) — *Stacked Generalization*
- Breiman (1996) — *Stacked Regressions* (trọng số không âm)
- scikit-learn User Guide — Stacked generalization